# Python Data Validation with Pydantic - Follow-Along Practice

This notebook contains all the practice examples and code snippets from the Pydantic chapter: base models, field constraints, custom validators, nested models, collection mappings, and data export serialization.

### Install Pydantic
Run the following cell to install Pydantic in your notebook environment.

In [ ]:
!pip install pydantic

### Basic BaseModel Example

#### Example: Schema Definition and Type Coercion
Declaring schemas by inheriting from `BaseModel`, demonstrating automatic casting, and handling `ValidationError` checks.

In [ ]:
from pydantic import BaseModel, ValidationError

# Define the schema
class User(BaseModel):
    id: int
    username: str
    email: str
    is_active: bool = True

# 1. Successful Instantiation with Type Coercion
user = User(id="123", username="alice", email="alice@example.com")
print("Coerced User ID:", user.id)
print("ID Type:", type(user.id))

#### Example: Validation Failures
Catching structured validation errors when type coercion is impossible.

In [ ]:
try:
    bad_user = User(id="not-an-int", username="bob", email="bob@example.com")
except ValidationError as e:
    print(e)

### Field Constraints

#### Example: Using Field limits
Declaring min/max ranges, descriptions, and boundary constraints on schemas.

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class Product(BaseModel):
    name: str = Field(min_length=2, max_length=50)
    price: float = Field(gt=0, description="Price must be greater than zero")
    stock: int = Field(default=0, ge=0)

# This will fail because price is negative
try:
    Product(name="Coffee", price=-2.50)
except ValidationError as e:
    print(e)

### Custom Field Validators

#### Example: Custom Password Validator
Defining conditional validations on fields using `@field_validator`.

In [ ]:
from pydantic import BaseModel, field_validator, ValidationError

class SignUp(BaseModel):
    username: str
    password: str

    @field_validator("password")
    @classmethod
    def password_must_be_strong(cls, v: str) -> str:
        if len(v) < 8:
            raise ValueError("Password must be at least 8 characters long")
        if not any(char.isdigit() for char in v):
            raise ValueError("Password must contain at least one digit")
        return v

try:
    SignUp(username="user1", password="weak")
except ValidationError as e:
    print(e)

### Nested Models

#### Example: Modeling Nested structures
Combining multiple schema models together in collection hierarchies.

In [ ]:
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    price: float

class Order(BaseModel):
    order_id: str
    items: list[Item]
    tax_rate: float = 0.08

# Instantiation using dictionaries
my_order = Order(
    order_id="ORD-100",
    items=[
        {"name": "Keyboard", "price": 49.99},
        {"name": "Mouse", "price": 19.99}
    ]
)

print("First item name:", my_order.items[0].name)

### Collections and Unions

#### Example: DeveloperProfile schema
Validating lists, sets, dictionaries, unions, and optional fields.

In [ ]:
from pydantic import BaseModel

class DeveloperProfile(BaseModel):
    name: str
    programming_languages: list[str]
    unique_certifications: set[str]
    monthly_contributions: dict[str, int]
    experience_years: int | float
    github_url: str | None = None

# Instantiation
dev = DeveloperProfile(
    name="Bob",
    programming_languages=["Python", "JavaScript"],
    unique_certifications={"AWS Certified", "Scrum Master", "AWS Certified"},
    monthly_contributions={"commits": "250", "prs": "12"},
    experience_years="5.5"
)

print("Certifications:", dev.unique_certifications)
print("Contributions:", dev.monthly_contributions)
print("Experience:", dev.experience_years)

### Exporting Data

#### Example: Dump to Dict and JSON String
Using built-in serializers `.model_dump()` and `.model_dump_json()`.

In [ ]:
user = User(id="123", username="alice", email="alice@example.com")

# Convert to dictionary
data_dict = user.model_dump()
print("Dictionary dump:", data_dict)

# Convert to JSON string
json_string = user.model_dump_json()
print("JSON string dump:", json_string)